In [1]:
import glob
import os
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [3]:
DATA_PATH = Path(r"D:\\ITC\\4th-Year\\Parallel-Distribute\\output\\ml_air_quality.parquet")   
OUTPUT_DIR = Path(r"D:\\ITC\\4th-Year\\Parallel-Distribute\\output\\model_results")   
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGETS = ["pm10_next_24h_mean", "pm25_next_24h_mean", "o3_next_24h_max"]
DROP_COLS = ["date", "station_name"]
CAT_COLS = ["city", "season"]
RANDOM_STATE = 42

In [4]:
def load_data(path):
    path = Path(path)
    if path.is_dir():
        parts = sorted(path.glob("part-*"))
        if not parts:
            raise FileNotFoundError(f"No part files found in {path}")
        if parts[0].suffix == ".parquet":
            return pd.concat([pd.read_parquet(f) for f in parts], ignore_index=True)
        return pd.concat([pd.read_csv(f) for f in parts], ignore_index=True)
    
df = load_data(DATA_PATH)
print(f"Loaded {len(df):,} rows × {len(df.columns)} columns")
df.head(3)


Loaded 2,349,095 rows × 51 columns


,date,latitude,longitude,station_name,city,wind_speed_u,wind_speed_v,dewpoint_temp,soil_temp,total_percipitation,...,pm25_lag3,no2_lag1,no2_lag2,no2_lag3,o3_lag1,o3_lag2,o3_lag3,pm10_next_24h_mean,pm25_next_24h_mean,o3_next_24h_max
0,2020-05-01 00:00:00,38.04972,23.757725,ATH-ENVICARE-2,athens,1.747032,-0.766625,9.860752,17.523380,0.000046,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12.185308,8.954384,104.890175
1,2020-05-01 01:00:00,38.04972,23.757725,ATH-ENVICARE-2,athens,1.733027,-0.779784,9.865142,17.262146,0.000000,...,NaN,12.66627,NaN,NaN,70.82164,NaN,NaN,12.095780,8.905776,104.890175
2,2020-05-01 02:00:00,38.04972,23.757725,ATH-ENVICARE-2,athens,1.739445,-0.836112,9.849197,16.999275,0.000000,...,NaN,10.21624,12.66627,NaN,65.67610,70.82164,NaN,12.010019,8.825388,104.890175


In [5]:
df = df.dropna(subset=TARGETS).reset_index(drop=True)
print(f"After dropping NaN targets: {len(df):,} rows")


After dropping NaN targets: 2,348,999 rows


In [6]:
y = df[TARGETS].values.astype(np.float32)
X = df.drop(columns=TARGETS + DROP_COLS, errors="ignore")

existing_cat = [c for c in CAT_COLS if c in X.columns]
X = pd.get_dummies(X, columns=existing_cat, drop_first=False)
feature_columns = X.columns.tolist()
print(f"Feature count: {len(feature_columns)}")
print(f"Categorical columns one-hot encoded: {existing_cat}")

X = X.values.astype(np.float32)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print(f"Train: {len(X_train):,}  Val: {len(X_val):,}")

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)


Feature count: 51
Categorical columns one-hot encoded: ['city', 'season']
Train: 1,879,199  Val: 469,800


In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

error: externally-managed-environment

× This environment is externally managed
╰─> This Python installation is managed by uv and should not be modified.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detailed specification.


In [2]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

2.12.1+cpu
False


In [7]:
class AirQualityMLP(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 3),   # pm10, pm25, o3
        )

    def forward(self, x):
        return self.network(x)


In [10]:
import torch

print("Torch:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

Torch: 2.12.1+cpu
CUDA version: None
CUDA available: False


In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AirQualityMLP(X_train_s.shape[1]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=5
)
criterion = nn.MSELoss()

def to_tensor(arr):
    return torch.tensor(arr, dtype=torch.float32, device=device)

MAX_EPOCHS = 20
PATIENCE = 10
BATCH_SIZE = 64

train_losses, val_losses = [], []
best_val_loss = float("inf")
patience_counter = 0

from tqdm.auto import tqdm

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()

    idxs = np.random.permutation(len(X_train_s))
    epoch_loss = 0.0

    pbar = tqdm(
        range(0, len(X_train_s), BATCH_SIZE),
        desc=f"Epoch {epoch}/{MAX_EPOCHS}",
        leave=False,
    )

    for i in pbar:
        batch_idxs = idxs[i:i+BATCH_SIZE]

        Xb = to_tensor(X_train_s[batch_idxs])
        yb = to_tensor(y_train[batch_idxs])

        optimizer.zero_grad()
        loss = criterion(model(Xb), yb)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * len(Xb)

        pbar.set_postfix(loss=f"{loss.item():.5f}")

KeyboardInterrupt: 

In [ ]:
model.eval()
with torch.no_grad():
    preds = model(to_tensor(X_val_s)).cpu().numpy()

rmse_per_target = np.sqrt(np.mean((preds - y_val) ** 2, axis=0))
target_names = ["PM10", "PM25", "O3"]
print("Validation RMSE per target:")
for name, rmse in zip(target_names, rmse_per_target):
    print(f"  {name}: {rmse:.4f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, (name, ax) in enumerate(zip(target_names, axes)):
    ax.scatter(y_val[:, i], preds[:, i], alpha=0.3, s=8)
    lims = [min(y_val[:, i].min(), preds[:, i].min()),
            max(y_val[:, i].max(), preds[:, i].max())]
    ax.plot(lims, lims, "r--", lw=1)
    ax.set(xlabel="Actual", ylabel="Predicted", title=name)
fig.tight_layout()
plt.savefig(OUTPUT_DIR / "mlp_val_scatter.png")
plt.show()


In [ ]:
model.eval().cpu()
dummy_input = torch.randn(1, X_train_s.shape[1])

torch.onnx.export(
    model,
    dummy_input,
    OUTPUT_DIR / "mlp_model.onnx",
    input_names=["features"],
    output_names=["pm10", "pm25", "o3"],
    dynamic_axes={
        "features": {0: "batch_size"},
        "pm10": {0: "batch_size"},
        "pm25": {0: "batch_size"},
        "o3": {0: "batch_size"},
    },
    opset_version=17,
)
print("Exported to", OUTPUT_DIR / "mlp_model.onnx")

# Check output with onnxruntime
import onnxruntime as ort
session = ort.InferenceSession(str(OUTPUT_DIR / "mlp_model.onnx"))
out = session.run(None, {"features": dummy_input.numpy()})
print("ONNX output shape:", [o.shape for o in out])
print("ONNX inference works ✓")


In [ ]:
import onnxruntime as ort
session = ort.InferenceSession(str(OUTPUT_DIR / "mlp_model.onnx"))
out = session.run(None, {"features": dummy_input.numpy()})
print("ONNX output shape:", [o.shape for o in out])
print("ONNX inference works ✓")
